# MASLD-LLM · 03 — Fine-tuning on Colab (Drive-synced edition)



Trains the model ladder from your Drive-synced folders. Sync mounts at

`/content/drive/Othercomputers/My Mac/` (read-only); checkpoints go to the

writable fallback if needed.



## Session 5 (smoke test): runtime = **L4**, model = **Qwen3-0.6B**



Paper recipe: LoRA r32 α32, ChatML, 3 epochs, assistant-only loss, neat packing.


## 0 · Mount + auto-discover


In [8]:
from google.colab import drive
drive.mount('/content/drive')
import os

SEARCH_ROOTS = ['/content/drive/Othercomputers/My Mac', '/content/drive/MyDrive']

def find_dir(target_name, search_roots=None, depth=6):
    if search_roots is None:
        search_roots = SEARCH_ROOTS
    matches = []
    for root in search_roots:
        direct = os.path.join(root, target_name)
        if os.path.isdir(direct):
            matches.append(direct)
    if matches:
        return matches
    def walk(path, d):
        if d > depth:
            return
        try:
            for entry in os.scandir(path):
                if entry.is_dir(follow_symlinks=False):
                    if entry.name == target_name:
                        matches.append(entry.path)
                    walk(entry.path, d + 1)
        except (PermissionError, OSError):
            pass
    for root in search_roots:
        if os.path.exists(root):
            walk(root, 0)
    return matches

cands = find_dir('masld-llm')
MASLD_LLM = min(cands, key=len) if cands else None
assert MASLD_LLM, 'masld-llm not found under Othercomputers/My Mac or MyDrive'
PROMPTS = os.path.join(MASLD_LLM, 'prompts')
print('PROMPTS =', PROMPTS)
print('corpora:', sorted(f for f in os.listdir(PROMPTS) if f.endswith('.jsonl.gz')))

def is_writable(path):
    try:
        probe = os.path.join(path, '.write_test')
        open(probe, 'w').close(); os.remove(probe)
        return True
    except Exception:
        return False

CKPT = (os.path.join(MASLD_LLM, 'checkpoints') if is_writable(MASLD_LLM)
        else '/content/drive/MyDrive/masld-llm-outputs/checkpoints')
os.makedirs(CKPT, exist_ok=True)
print('CKPT (writable) =', CKPT)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
PROMPTS = /content/drive/Othercomputers/My Mac/masld-llm/prompts
corpora: ['MB-10xx_liver_train.jsonl.gz', 'MB-11xx_external_train.jsonl.gz', 'MB-20xx_liver_test.jsonl.gz', 'MB-30xx_dnam_test.jsonl.gz', 'MB-30xx_dnam_train.jsonl.gz', 'MB-61xx_msigdb_train.jsonl.gz', 'MB-62xx_drugs_train.jsonl.gz', 'MB-63xx_drugcentral_affinity_train.jsonl.gz', 'MB-70xx_dge_direction_train.jsonl.gz', 'MB-8xxx_fujiwara_train.jsonl.gz', 'MB-9xxx_fujiwara_KU_test.jsonl.gz']
CKPT (writable) = /content/drive/Othercomputers/My Mac/masld-llm/checkpoints


## 1 · Install + convert all seven train corpora


In [9]:
%cd /content
!git clone --depth 1 https://github.com/hiyouga/LLaMA-Factory.git
%cd LLaMA-Factory
!pip -q install -e ".[torch,metrics]" bitsandbytes peft transformers accelerate

import gzip, json, os

# --- Recovery block for missing global variables (PROMPTS) ---
if 'PROMPTS' not in globals():
    print("PROMPTS not found in workspace. Re-discovering environment...")
    SEARCH_ROOTS = ['/content/drive/Othercomputers/My Mac', '/content/drive/MyDrive']
    def find_dir(target_name, search_roots=None, depth=6):
        if search_roots is None:
            search_roots = SEARCH_ROOTS
        matches = []
        for root in search_roots:
            direct = os.path.join(root, target_name)
            if os.path.isdir(direct):
                matches.append(direct)
        if matches:
            return matches
        def walk(path, d):
            if d > depth:
                return
            try:
                for entry in os.scandir(path):
                    if entry.is_dir(follow_symlinks=False):
                        if entry.name == target_name:
                            matches.append(entry.path)
                        walk(entry.path, d + 1)
            except (PermissionError, OSError):
                pass
        for root in search_roots:
            if os.path.exists(root):
                walk(root, 0)
        return matches

    cands = find_dir('masld-llm')
    MASLD_LLM = min(cands, key=len) if cands else None
    assert MASLD_LLM, 'masld-llm not found under Othercomputers/My Mac or MyDrive. Please run the Mount cell first.'
    PROMPTS = os.path.join(MASLD_LLM, 'prompts')
    print('PROMPTS recovered =', PROMPTS)

def to_alpaca(in_path, out_path):
    rows = []
    with gzip.open(in_path, 'rt') as f:
        first = json.loads(f.readline())
        if '__meta__' not in first:
            rows.append(first)
        for l in f:
            rows.append(json.loads(l))
    out = [dict(instruction=r['prompt'], output=str(r['answer'])) for r in rows]
    json.dump(out, open(out_path, 'w'), indent=0)
    print(out_path, len(out))

to_alpaca(f'{PROMPTS}/MB-10xx_liver_train.jsonl.gz',      '/content/sft_liver.json')
to_alpaca(f'{PROMPTS}/MB-11xx_external_train.jsonl.gz',   '/content/sft_external.json')
to_alpaca(f'{PROMPTS}/MB-61xx_msigdb_train.jsonl.gz',     '/content/sft_knowledge.json')
to_alpaca(f'{PROMPTS}/MB-62xx_drugs_train.jsonl.gz',      '/content/sft_drugs.json')
to_alpaca(f'{PROMPTS}/MB-63xx_drugcentral_affinity_train.jsonl.gz', '/content/sft_drugs2.json')
to_alpaca(f'{PROMPTS}/MB-70xx_dge_direction_train.jsonl.gz', '/content/sft_dge.json')
to_alpaca(f'{PROMPTS}/MB-8xxx_fujiwara_train.jsonl.gz',   '/content/sft_fujiwara.json')
to_alpaca(f'{PROMPTS}/MB-30xx_dnam_train.jsonl.gz',       '/content/sft_dnam.json')

/content
fatal: destination path 'LLaMA-Factory' already exists and is not an empty directory.
/content/LLaMA-Factory
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Installing backend dependencies ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for llamafactory (pyproject.toml) ... done
/content/sft_liver.json 3928
/content/sft_external.json 5575
/content/sft_knowledge.json 2701
/content/sft_drugs.json 30000
/content/sft_drugs2.json 39632
/content/sft_dge.json 8088
/content/sft_fujiwara.json 204
/content/sft_dnam.json 73


In [10]:
# v3 RECIPE (token-budgeted): v2's blind x4 liver upsample tripled long-prompt
# tokens (~20h train). The v1 failure was FORMAT volume: multiclass/binary/reg
# had 135 examples each (0.4%). Fix: upsample single-profile formats x16,
# keep pairwise x1 (it already works, and dominates liver), cap drugs 10k.
import json, os, random
rs = random.Random(20260929)

def load(p):
    return json.load(open(p)) if os.path.exists(p) else []

liver = load('/content/sft_liver.json')            # 3928: 3388 pairwise + 540 single-profile
external = load('/content/sft_external.json')      # 5575 (mixed formats)
fujiwara = load('/content/sft_fujiwara.json')      # 204 single-profile
knowledge = load('/content/sft_knowledge.json')    # 2701
dge = load('/content/sft_dge.json')                # 8088
drugs1 = load('/content/sft_drugs.json')           # 30000
drugs2 = load('/content/sft_drugs2.json')          # 39632
dnam = load('/content/sft_dnam.json')              # 73

# split liver by format: pairwise prompts contain two patient blocks
def is_pairwise(ex):
    return 'Patient A:' in ex['instruction']
liver_single = [e for e in liver if not is_pairwise(e)]   # 540
liver_pair = [e for e in liver if is_pairwise(e)]          # 3388
ext_single = [e for e in external if not is_pairwise(e)]
ext_pair = [e for e in external if is_pairwise(e)]

def upsample(rows, factor):
    if not rows or factor <= 1:
        return list(rows)
    out = list(rows)
    target = len(rows) * factor
    while len(out) < target:
        out += rs.sample(rows, min(len(rows), target - len(out)))
    return out[:target]

# x16 the failing single-profile formats; pairwise x1 (already 0.47 at v1)
single = (upsample(liver_single, 16) + ext_single
          + upsample(fujiwara, 8) + upsample(dnam, 20))
pair = liver_pair + ext_pair
drugs = rs.sample(drugs1 + drugs2, 10000)
know = rs.sample(knowledge + dge, 8000)

mixture = single + pair + drugs + know
rs.shuffle(mixture)
json.dump(mixture, open('/content/sft_multitask.json', 'w'), indent=0)
print(f'single-profile: {len(liver_single)}x16 + {len(ext_single)} + fujiwara/dnam ups = {len(single)}')
print(f'pairwise: {len(pair)} | drugs {len(drugs)} | knowledge {len(know)}')
print(f'FINAL MIXTURE: {len(mixture)} (single {100*len(single)/len(mixture):.0f}%, pair {100*len(pair)/len(mixture):.0f}%, drugs {100*len(drugs)/len(mixture):.0f}%, know {100*len(know)/len(mixture):.0f}%)')
print(f'expected train time ~ 2 epochs x ~26M tokens/epoch ~ 12h (v1: 3 x 16M = 11h)')

single-profile: 540x16 + 1003 + fujiwara/dnam ups = 12735
pairwise: 7960 | drugs 10000 | knowledge 8000
FINAL MIXTURE: 38695 (single 33%, pair 21%, drugs 26%, know 21%)
expected train time ~ 2 epochs x ~26M tokens/epoch ~ 12h (v1: 3 x 16M = 11h)


## 2 · Register dataset + 0.6B config


In [11]:
ds_entry = '''{
  "masld_sft": {
    "file_name": "/content/sft_multitask.json",
    "columns": {"prompt": "instruction", "response": "output"}
  }
}'''
open('data/dataset_info.json', 'w').write(ds_entry)

yaml_06b = '''
model_name_or_path: Qwen/Qwen3-0.6B
trust_remote_code: true
stage: sft
do_train: true
finetuning_type: lora
lora_rank: 32
lora_alpha: 32
lora_dropout: 0.0
dataset: masld_sft
template: qwen
cutoff_len: 4096
packing: true
neat_packing: true
max_samples: 100000
overwrite_cache: true
preprocessing_num_workers: 16
output_dir: ''' + CKPT + '''/qwen3-0.6b-masld-sft
logging_steps: 10
save_steps: 0.25
plot_loss: true
per_device_train_batch_size: 1
gradient_accumulation_steps: 32
learning_rate: 1.0e-4
num_train_epochs: 2.0
lr_scheduler_type: cosine
warmup_ratio: 0.1
weight_decay: 0.01
bf16: true
gradient_checkpointing: true
enable_liger_kernel: true
'''
open('/content/qwen06b_sft.yaml', 'w').write(yaml_06b)
print('config written; CKPT =', CKPT)
print('v3: 2 epochs, mixture ~34k, format-aware upsampling')

config written; CKPT = /content/drive/Othercomputers/My Mac/masld-llm/checkpoints
v3: 2 epochs, mixture ~34k, format-aware upsampling


In [ ]:
import os

# Fix CUDA version mismatch between PyTorch (cu130) and TorchAudio (cu128)
!pip uninstall -y torchaudio
!pip install torchaudio --index-url https://download.pytorch.org/whl/cu130

# Liger kernels (fused cross-entropy) - required to avoid the 9.3GB vocab-logits OOM
!pip -q install liger-kernel

# Session 5 smoke test: watch loss drop 4-5 -> <2 within 50 steps
!DISABLE_VERSION_CHECK=1 python -m llamafactory.cli train /content/qwen06b_sft.yaml

Found existing installation: torchaudio 2.11.0+cu130
Uninstalling torchaudio-2.11.0+cu130:
  Successfully uninstalled torchaudio-2.11.0+cu130
Looking in indexes: https://download.pytorch.org/whl/cu130
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 63.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 900.2/900.2 kB 53.5 MB/s eta 0:00:00
[WARNING|2026-10-04 12:17:03] llamafactory.extras.misc:155 >> Version checking has been disabled, may lead to unexpected behaviors.
/usr/local/lib/python3.13/dist-packages/jieba/__init__.py:44: SyntaxWarning: invalid escape sequence '\.'
  re_han_default = re.compile("([\u4E00-\u9FD5a-zA-Z0-9+#&\._%\-]+)", re.U)
/usr/local/lib/python3.13/dist-packages/jieba/__init__.py:46: SyntaxWarning: invalid escape sequence '\s'
  re_skip_default = re.compile("(\r\n|\s)", re.U)
/usr/local/lib/python3.13/dist-packages/jieba/finalseg/__init__.py:78: SyntaxWarning: invalid escape sequence '\.'
  re_skip = re.compile("([a-zA-Z0-9]+(?:\.\d+)?%?)")
/us

## 3 · Evaluate on the frozen test (first C2-contrast row)


In [ ]:
# Merge the LoRA adapter into the base model BEFORE installing vLLM.
# Primary: llamafactory export (yaml, output captured). Fallback: direct PeFT
# merge_and_unload() - no llamafactory dependency, works with any checkpoint dir.
import subprocess, sys, os, glob, shutil

CKPT_SFT = f'{CKPT}/qwen3-0.6b-masld-sft'

adapter = None
if os.path.exists(os.path.join(CKPT_SFT, 'adapter_model.safetensors')):
    adapter = CKPT_SFT
else:
    ckpts = sorted(glob.glob(f'{CKPT_SFT}/checkpoint-*'),
                   key=lambda p: int(p.rsplit('-', 1)[-1]))
    if ckpts:
        adapter = ckpts[-1]
assert adapter, f'No adapter found in {CKPT_SFT} - re-run cell 8 (training) first.'
if adapter != CKPT_SFT:
    print('NOTE: root adapter missing => training did not finish.')
    print('      Merging the LATEST checkpoint. Check cell 8 output to see why it stopped;')
    print('      re-run cell 8 for the final adapter if it crashed.')
print('merging adapter:', adapter)

MERGED = '/content/qwen3-0.6b-masld-merged'   # local disk: fast, avoids Drive stalls

# ---------- primary: llamafactory export (errors now visible) ----------
export_yaml = f'''
model_name_or_path: Qwen/Qwen3-0.6B
adapter_name_or_path: {adapter}
template: qwen
finetuning_type: lora
export_dir: {MERGED}
export_size: 2
export_device: cpu
export_legacy_format: false
'''
open('/content/export_06b.yaml', 'w').write(export_yaml)

ok_export = False
try:
    env = dict(os.environ, DISABLE_VERSION_CHECK='1')
    result = subprocess.run(
        [sys.executable, '-m', 'llamafactory.cli', 'export', '/content/export_06b.yaml'],
        env=env, capture_output=True, text=True, timeout=1800)
    if result.stdout:
        print('--- llamafactory export stdout (tail) ---')
        print(result.stdout[-2500:])
    if result.returncode == 0 and os.path.exists(f'{MERGED}/config.json'):
        ok_export = True
    else:
        print('--- llamafactory export FAILED, stderr (tail) ---')
        print(result.stderr[-2500:])
except Exception as e:
    print('llamafactory export raised:', e)

# ---------- fallback: direct PeFT merge (always works) ----------
if not ok_export:
    print('=== falling back to direct PeFT merge_and_unload() ===')
    import torch
    from transformers import AutoModelForCausalLM, AutoTokenizer
    from peft import PeftModel
    base = AutoModelForCausalLM.from_pretrained(
        'Qwen/Qwen3-0.6B', torch_dtype=torch.bfloat16)
    model = PeftModel.from_pretrained(base, adapter)
    model = model.merge_and_unload()
    model.save_pretrained(MERGED, safe_serialization=True)
    AutoTokenizer.from_pretrained('Qwen/Qwen3-0.6B').save_pretrained(MERGED)
    print('PeFT merge complete ->', MERGED)

# ---------- persist to Drive (best-effort; ~1.2 GB) ----------
try:
    dst = f'{CKPT}/qwen3-0.6b-masld-merged'
    if not os.path.exists(dst):
        shutil.copytree(MERGED, dst)
    print('persisted to Drive:', dst)
except Exception as e:
    print('Drive copy skipped (model is safe at', MERGED, '):', e)

merging adapter: /content/drive/Othercomputers/My Mac/masld-llm/checkpoints/qwen3-0.6b-masld-sft
--- llamafactory export stdout (tail) ---
[WARNING|2026-10-04 17:17:09] llamafactory.extras.misc:155 >> Version checking has been disabled, may lead to unexpected behaviors.
[INFO|2026-10-04 17:17:16] llamafactory.model.model_utils.kv_cache:144 >> KV cache is enabled for faster generation.
[INFO|2026-10-04 17:17:21] llamafactory.model.model_utils.attention:144 >> Using torch SDPA for faster training and inference.
[INFO|2026-10-04 17:17:25] llamafactory.model.adapter:144 >> Merged 1 adapter(s).
[INFO|2026-10-04 17:17:25] llamafactory.model.adapter:144 >> Loaded adapter(s): /content/drive/Othercomputers/My Mac/masld-llm/checkpoints/qwen3-0.6b-masld-sft
[INFO|2026-10-04 17:17:25] llamafactory.model.loader:144 >> all params: 596,049,920
[INFO|2026-10-04 17:17:25] llamafactory.train.tuner:144 >> Convert model dtype to: torch.bfloat16.
[INFO|2026-10-04 17:17:26] llamafactory.train.tuner:144 >> O

### vLLM install + evaluation


In [ ]:
# NOTE: vLLM 0.30 on Colab may fail with 'libcudart.so.13 not found' - the base
# image ships CUDA-12 RAPIDS libs while vllm links CUDA 13. This cell tries the
# vLLM fast path (with a preload fix) and falls back to transformers batch
# generation, which needs no extra CUDA libs and is fast enough for a 0.6B model.
import gzip, json, random, re, os, glob

model_path = ('/content/qwen3-0.6b-masld-merged'
              if os.path.exists('/content/qwen3-0.6b-masld-merged')
              else f'{CKPT}/qwen3-0.6b-masld-merged')
assert os.path.exists(f'{model_path}/config.json'), f'merged model missing at {model_path} - re-run cell 10'
print('evaluating model:', model_path)

def parse_answer(text, task):
    t = (text or '').strip().lower()
    if task.endswith(('001', '201')):
        if t.startswith('yes') or 'answer: yes' in t or 'answer is yes' in t: return 'yes'
        if t.startswith('no') or 'answer: no' in t or 'answer is no' in t: return 'no'
        return None
    if task.endswith(('003', '203')):
        if re.search(r'\banswer(?: is|:) a\b', t) or t.startswith('a'): return 'A'
        if re.search(r'\banswer(?: is|:) b\b', t) or t.startswith('b'): return 'B'
        return None
    m = re.search(r'\bf([0-4])\b', t)
    if m: return f'F{m.group(1)}'
    m = re.search(r'([0-9]+(?:\.[0-9]+)?)', t)
    if m:
        try: return float(m.group(1))
        except ValueError: return None
    return None

rng = random.Random(20260929)
rows = []
with gzip.open(f'{PROMPTS}/MB-20xx_liver_test.jsonl.gz', 'rt') as f:
    f.readline()
    for line in f:
        rows.append(json.loads(line))
by_task = {}
for r in rows:
    by_task.setdefault(r['task'], []).append(r)
sample = []
for task, items in sorted(by_task.items()):
    rng.shuffle(items)
    sample.extend(items[:300])
print('eval prompts:', len(sample))

raws = None

# ---------- path 1: vLLM with libcudart preload fix ----------
try:
    try:
        from vllm import LLM, SamplingParams
    except ImportError as e:
        if 'libcudart' not in str(e):
            raise
        import ctypes, subprocess
        cands = (glob.glob('/usr/local/lib/python3.*/dist-packages/nvidia/*/lib/libcudart.so.13')
                 + glob.glob('/usr/local/cuda*/lib64/libcudart.so.13'))
        if not cands:
            cands = subprocess.run(
                ['find', '/usr/local/lib/python3.13/dist-packages', '-name', 'libcudart.so.13'],
                capture_output=True, text=True).stdout.split()
        if not cands:
            raise RuntimeError('libcudart.so.13 not found anywhere')
        ctypes.CDLL(cands[0], mode=ctypes.RTLD_GLOBAL)
        from vllm import LLM, SamplingParams
    llm = LLM(model=model_path, gpu_memory_utilization=0.9)
    sp = SamplingParams(temperature=0.0, top_p=1.0, max_tokens=200)
    outs = llm.generate([r['prompt'] for r in sample], sp)
    raws = [o.outputs[0].text for o in outs]
    print('vLLM path used')
except Exception as e:
    print('vLLM path failed:', str(e)[:200])

# ---------- path 2: transformers batched greedy generation ----------
if raws is None:
    print('=== using transformers fallback ===')
    import torch
    from transformers import AutoModelForCausalLM, AutoTokenizer
    tok = AutoTokenizer.from_pretrained(model_path, padding_side='left')
    if tok.pad_token is None:
        tok.pad_token = tok.eos_token
    model = AutoModelForCausalLM.from_pretrained(
        model_path, torch_dtype=torch.bfloat16, device_map='cuda')
    model.eval()
    raws = []
    BATCH = 8
    i = 0
    while i < len(sample):
        batch = sample[i:i+BATCH]
        try:
            enc = tok([r['prompt'] for r in batch], return_tensors='pt',
                      padding=True, truncation=True, max_length=4096).to('cuda')
            with torch.no_grad():
                gen = model.generate(**enc, max_new_tokens=200, do_sample=False,
                                     pad_token_id=tok.pad_token_id)
            for j in range(len(batch)):
                raws.append(tok.decode(gen[j][enc['input_ids'].shape[1]:],
                                       skip_special_tokens=True))
            i += len(batch)
        except torch.cuda.OutOfMemoryError:
            torch.cuda.empty_cache()
            if BATCH == 1:
                raise
            BATCH = max(1, BATCH // 2)
            print(f'OOM -> batch reduced to {BATCH}, restarting this chunk')
        if (i // BATCH) % 10 == 0 and i % 80 < BATCH:
            print(f'  {i}/{len(sample)}', flush=True)
    print('transformers path complete')

# ---------- archive (identical format to frontier runs) ----------
runs_dir = (os.path.join(MASLD_LLM, 'results', 'frontier_runs')
            if os.access(os.path.join(MASLD_LLM, 'results'), os.W_OK)
            else '/content/drive/MyDrive/masld-llm-outputs/frontier_runs')
os.makedirs(runs_dir, exist_ok=True)
run_path = runs_dir + '/qwen3-0.6b-masld__MB-20xx.jsonl.gz'
with gzip.open(run_path, 'wt') as f:
    for r, raw in zip(sample, raws):
        f.write(json.dumps(dict(task=r['task'], sample=r['sample'], answer=r['answer'],
                                raw=raw, error=None, model_version='qwen3-0.6b-masld-sft',
                                parsed=parse_answer(raw, r['task']))) + chr(10))
print('archived ->', run_path)

evaluating model: /content/qwen3-0.6b-masld-merged
eval prompts: 972
vLLM path failed: No module named 'vllm'
=== using transformers fallback ===


[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/310 [00:00<?, ?it/s]

  80/972
  160/972
  240/972
  320/972
  400/972
  480/972
  560/972
  640/972
  720/972
  800/972
  880/972
  960/972
transformers path complete
archived -> /content/drive/Othercomputers/My Mac/masld-llm/results/frontier_runs/qwen3-0.6b-masld__MB-20xx.jsonl.gz


### ONE-SHOT verify + eval (run this instead of the cell above if outputs cycle)


In [ ]:
# ONE-SHOT VERIFY + EVAL: settles stale-vs-genuine and produces a valid scored run.
# Run this single cell after the merge cell. It:
#   1. fingerprints the /content merged model against the finished adapter
#   2. re-runs the eval with generation that STOPS at the first newline
#      (the cycling is a packing artifact: the model learned to continue past
#      the answer, so we must stop it ourselves)
#   3. archives + prints scores inline
import os, glob, gzip, json, random, re, hashlib, shutil
import numpy as np

# --- Recovery block for missing global variables (CKPT, PROMPTS, MASLD_LLM) ---
if 'CKPT' not in globals() or 'PROMPTS' not in globals() or 'MASLD_LLM' not in globals():
    print("Re-discovering environment variables...")
    SEARCH_ROOTS = ['/content/drive/Othercomputers/My Mac', '/content/drive/MyDrive']
    def find_dir(target_name, search_roots=None, depth=6):
        if search_roots is None:
            search_roots = SEARCH_ROOTS
        matches = []
        for root in search_roots:
            direct = os.path.join(root, target_name)
            if os.path.isdir(direct):
                matches.append(direct)
        if matches:
            return matches
        def walk(path, d):
            if d > depth:
                return
            try:
                for entry in os.scandir(path):
                    if entry.is_dir(follow_symlinks=False):
                        if entry.name == target_name:
                            matches.append(entry.path)
                        walk(entry.path, d + 1)
            except (PermissionError, OSError):
                pass
        for root in search_roots:
            if os.path.exists(root):
                walk(root, 0)
        return matches

    cands = find_dir('masld-llm')
    MASLD_LLM = min(cands, key=len) if cands else None
    assert MASLD_LLM, 'masld-llm not found under Othercomputers/My Mac or MyDrive'
    PROMPTS = os.path.join(MASLD_LLM, 'prompts')

    def is_writable(path):
        try:
            probe = os.path.join(path, '.write_test')
            open(probe, 'w').close(); os.remove(probe)
            return True
        except Exception:
            return False

    CKPT = (os.path.join(MASLD_LLM, 'checkpoints') if is_writable(MASLD_LLM)
            else '/content/drive/MyDrive/masld-llm-outputs/checkpoints')

CKPT_SFT = f'{CKPT}/qwen3-0.6b-masld-sft'
MERGED = '/content/qwen3-0.6b-masld-merged'
root_adapter = os.path.join(CKPT_SFT, 'adapter_model.safetensors')
merged_file = os.path.join(MERGED, 'model.safetensors')

# Fallback: if local merged folder is missing, try to restore from Drive
if not os.path.exists(merged_file):
    drive_merged = f'{CKPT}/qwen3-0.6b-masld-merged'
    if os.path.exists(os.path.join(drive_merged, 'model.safetensors')):
        print(f"Restoring merged model from Drive backup to {MERGED}...")
        os.makedirs(MERGED, exist_ok=True)
        for item in os.listdir(drive_merged):
            s = os.path.join(drive_merged, item)
            d = os.path.join(MERGED, item)
            if os.path.isdir(s):
                shutil.copytree(s, d, dirs_exist_ok=True)
            else:
                shutil.copy2(s, d)

assert os.path.exists(root_adapter), 'finished root adapter missing'
assert os.path.exists(merged_file), f'merged model missing at {MERGED} - run the merge cell first'

# ---- 1. fingerprint check: merge MUST be newer than the finished adapter ----
t_merged = os.path.getmtime(merged_file)
t_adapter = os.path.getmtime(root_adapter)
import datetime
print(f'merged model:   {datetime.datetime.fromtimestamp(t_merged)}')
print(f'root adapter:   {datetime.datetime.fromtimestamp(t_adapter)}')
if t_merged < t_adapter:
    print('*** STALE MERGE CONFIRMED - rebuilding now ***')
    import subprocess, sys
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-U',
                    'torchao>=0.16'], check=False)
    import torch
    from transformers import AutoModelForCausalLM, AutoTokenizer
    from peft import PeftModel
    base = AutoModelForCausalLM.from_pretrained('Qwen/Qwen3-0.6B', torch_dtype=torch.bfloat16)
    model = PeftModel.from_pretrained(base, CKPT_SFT)
    model = model.merge_and_unload()
    model.save_pretrained(MERGED, safe_serialization=True)
    AutoTokenizer.from_pretrained('Qwen/Qwen3-0.6B').save_pretrained(MERGED)
    print('rebuilt fresh ->', MERGED)
else:
    print('merge is FRESH (newer than adapter) - identical outputs are genuine model behavior')

# ---- 2. eval with newline-stopping generation ----
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

tok = AutoTokenizer.from_pretrained(MERGED, padding_side='left')
if tok.pad_token is None:
    tok.pad_token = tok.eos_token
model = AutoModelForCausalLM.from_pretrained(MERGED, torch_dtype=torch.bfloat16, device_map='cuda')
model.eval()

class StopAtNewline:
    def __init__(self):
        self.newlines = []
    def __call__(self, input_ids, scores, **kw):
        # stop when the batch's sequences contain a newline in the generated part
        for i, seq in enumerate(input_ids):
            gen = seq[kw.get('prompt_len', 0):] if 'prompt_len' in kw else seq[-1:]
        # simpler: rely on post-truncation below; use EOS + max 30 tokens
        return False

# simplest robust approach: generate few tokens, truncate at first newline
rng = random.Random(20260929)
rows = []
with gzip.open(f'{PROMPTS}/MB-20xx_liver_test.jsonl.gz', 'rt') as f:
    f.readline()
    for line in f:
        rows.append(json.loads(line))
by_task = {}
for r in rows:
    by_task.setdefault(r['task'], []).append(r)
sample = []
for task, items in sorted(by_task.items()):
    rng.shuffle(items)
    sample.extend(items[:300])
print('eval prompts:', len(sample))

def parse_first(text, task):
    t = (text or '').strip()
    fl = t.split('\n')[0].strip().lower()
    if task.endswith(('001', '201')):
        if fl.startswith('yes'): return 'yes'
        if fl.startswith('no'): return 'no'
        return None
    if task.endswith(('003', '203')):
        if fl.startswith('a') and not fl.startswith('adv'): return 'A'
        if fl.startswith('b'): return 'B'
        return None
    m = re.search(r'\bf([0-4])\b', fl)
    if m: return f'F{m.group(1)}'
    m = re.match(r'([0-9]+(?:\.[0-9]+)?)', fl)
    if m:
        try: return float(m.group(1))
        except ValueError: return None
    return None

raws = []
BATCH = 16
i = 0
while i < len(sample):
    batch = sample[i:i+BATCH]
    try:
        enc = tok([r['prompt'] for r in batch], return_tensors='pt',
                  padding=True, truncation=True, max_length=4096).to('cuda')
        with torch.no_grad():
            gen = model.generate(**enc, max_new_tokens=30, do_sample=False,
                                 pad_token_id=tok.pad_token_id,
                                 eos_token_id=tok.eos_token_id)
        for j in range(len(batch)):
            text = tok.decode(gen[j][enc['input_ids'].shape[1]:], skip_special_tokens=True)
            raws.append(text.split('\n')[0])   # first line ONLY: kills the cycling artifact
        i += len(batch)
    except torch.cuda.OutOfMemoryError:
        torch.cuda.empty_cache()
        if BATCH == 1: raise
        BATCH = max(1, BATCH // 2)
        print(f'OOM -> batch {BATCH}')
    if i % 160 < BATCH:
        print(f'  {i}/{len(sample)}', flush=True)
print('generation complete (max_new_tokens=30, first-line truncation)')

# ---- 3. archive + score inline ----
runs_dir = (os.path.join(MASLD_LLM, 'results', 'frontier_runs')
            if os.access(os.path.join(MASLD_LLM, 'results'), os.W_OK)
            else '/content/drive/MyDrive/masld-llm-outputs/frontier_runs')
os.makedirs(runs_dir, exist_ok=True)
run_path = runs_dir + '/qwen3-0.6b-masld__MB-20xx.jsonl.gz'
with gzip.open(run_path, 'wt') as f:
    for r, raw in zip(sample, raws):
        f.write(json.dumps(dict(task=r['task'], sample=r['sample'], answer=r['answer'],
                                raw=raw, error=None, model_version='qwen3-0.6b-masld-sft-v2-eval',
                                parsed=parse_first(raw, r['task']))) + chr(10))
print('archived ->', run_path)

# inline scores
by_t = {}
for r, raw in zip(sample, raws):
    p = parse_first(raw, r['task'])
    by_t.setdefault(r['task'], []).append((p, r['answer']))
print()
print('=== qwen3-0.6b-masld (fresh eval, newline-stopped) ===')
for t, pairs in sorted(by_t.items()):
    parsed = [p for p, g in pairs]; gold = [g for p, g in pairs]
    valid = sum(1 for p in parsed if p is not None)
    if t in ('MB-2001',):
        acc = sum(1 for p, g in pairs if p == g) / len(pairs)
        print(f'  {t}: acc={acc:.3f} validity={valid/len(pairs):.3f}')
    elif t == 'MB-2003':
        acc = sum(1 for p, g in pairs if p == g) / len(pairs)
        print(f'  {t}: concordance={acc:.3f} validity={valid/len(pairs):.3f}')
    elif t == 'MB-2002':
        acc = sum(1 for p, g in pairs if p == g) / len(pairs)
        print(f'  {t}: acc={acc:.3f} validity={valid/len(pairs):.3f}')
    else:
        nums = [(p, float(g)) for p, g in pairs if isinstance(p, (int, float))]
        if nums:
            mae = float(np.mean([abs(p - g) for p, g in nums]))
            from scipy.stats import spearmanr
            rho = float(spearmanr([g for p, g in nums], [p for p, g in nums]).statistic)
            print(f'  {t}: mae={mae:.3f} rho={rho:.3f} validity={valid/len(pairs):.3f}')
        else:
            print(f'  {t}: no parseable numbers, validity={valid/len(pairs):.3f}')
print()
print('references: majority 0.582 | nemotron binary 0.582/multi 0.231/pair 0.497/MAE 2.12 | ML EN multi 0.253/MAE 1.76')

merged model:   2026-10-04 17:17:26.754001
root adapter:   2026-10-04 17:16:50
merge is FRESH (newer than adapter) - identical outputs are genuine model behavior


Loading weights:   0%|          | 0/310 [00:00<?, ?it/s]

eval prompts: 972
  160/972
  320/972
  480/972
  640/972
  800/972
  960/972
  972/972
generation complete (max_new_tokens=30, first-line truncation)
archived -> /content/drive/Othercomputers/My Mac/masld-llm/results/frontier_runs/qwen3-0.6b-masld__MB-20xx.jsonl.gz

=== qwen3-0.6b-masld (fresh eval, newline-stopped) ===
  MB-2001: acc=0.405 validity=0.601
  MB-2002: acc=0.000 validity=0.000
  MB-2003: concordance=0.593 validity=1.000
  MB-2005: mae=1.494 rho=0.011 validity=1.000
  MB-2006: mae=595238095238095200191512576.000 rho=-0.024 validity=1.000

references: majority 0.582 | nemotron binary 0.582/multi 0.231/pair 0.497/MAE 2.12 | ML EN multi 0.253/MAE 1.76


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 4 · Ladder continuation

- **1.7B**: same config, `model_name_or_path: Qwen/Qwen3-1.7B`.

- **9B two-stage QLoRA**: A100 40GB (Pro+). Stage A = KG/drug text, 1 epoch, lr 2e-5;

  stage B = this config from the stage-A adapter, lr 1e-4, `quantization_bit: 4`.

- After each run: sync home, then locally `python3 scripts/score_frontier.py`.


## 5 · 1.7B run (v4 recipe: class-balanced)

Run the three cells below in order after the globals (cell 1) and corpus conversion (cell 2).
Training takes ~10–14 h on L4; checkpoints save to Drive and resume automatically.


In [ ]:
# ===== 1.7B RUN (v4 recipe) =====
# v3 results: pairwise 0.593 (beats frontier), multiclass 0.208 but collapsed to
# F0/F1 (early-stage bias from imbalanced upsampling), binary 0.405.
# v4: class-balanced multiclass (F2-F4 upweighted), binary yes upweighted,
# NAS upweighted. Model: Qwen3-1.7B. Expected train time ~10-14h on L4.
import os, json, random

# --- global recovery (if runtime was reset) ---
if 'CKPT' not in globals() or 'PROMPTS' not in globals():
    import glob as _glob
    SEARCH_ROOTS = ['/content/drive/Othercomputers/My Mac', '/content/drive/MyDrive']
    def find_dir(target_name, search_roots=None, depth=6):
        if search_roots is None: search_roots = SEARCH_ROOTS
        matches = []
        for root in search_roots:
            direct = os.path.join(root, target_name)
            if os.path.isdir(direct): matches.append(direct)
        if matches: return matches
        def walk(path, d):
            if d > depth: return
            try:
                for entry in os.scandir(path):
                    if entry.is_dir(follow_symlinks=False):
                        if entry.name == target_name: matches.append(entry.path)
                        walk(entry.path, d + 1)
            except (PermissionError, OSError): pass
        for root in search_roots:
            if os.path.exists(root): walk(root, 0)
        return matches
    _c = find_dir('masld-llm')
    MASLD_LLM = min(_c, key=len) if _c else None
    assert MASLD_LLM, 'masld-llm not found'
    PROMPTS = os.path.join(MASLD_LLM, 'prompts')
    CKPT = os.path.join(MASLD_LLM, 'checkpoints')
    print('recovered: MASLD_LLM =', MASLD_LLM)

rs = random.Random(20260929)

def load(p):
    return json.load(open(p)) if os.path.exists(p) else []

liver = load('/content/sft_liver.json')
external = load('/content/sft_external.json')
fujiwara = load('/content/sft_fujiwara.json')
knowledge = load('/content/sft_knowledge.json')
dge = load('/content/sft_dge.json')
drugs1 = load('/content/sft_drugs.json')
drugs2 = load('/content/sft_drugs2.json')
dnam = load('/content/sft_dnam.json')

def is_pairwise(ex):
    return 'Patient A:' in ex['instruction']
liver_single = [e for e in liver if not is_pairwise(e)]
liver_pair = [e for e in liver if is_pairwise(e)]
ext_single = [e for e in external if not is_pairwise(e)]
ext_pair = [e for e in external if is_pairwise(e)]

# --- v4: class-aware upsampling of liver singles ---
# multiclass answers: balance F0..F4 to ~equal (train dist F0:32 F1:50 F2:23 F3:23 F4:7)
STAGE_W = {'F0': 2, 'F1': 1, 'F2': 2, 'F3': 2, 'F4': 7}
def task_kind(ex):
    q = ex['instruction']
    if 'Assign the fibrosis stage' in q or 'Which fibrosis stage' in q: return 'multi'
    if 'advanced fibrosis' in q: return 'binary'
    if 'NAFLD Activity Score' in q or 'exact NAS' in q: return 'nas'
    if 'age in years' in q or 'chronological age' in q: return 'age'
    return 'fibreg'

def upsample_weighted(rows, base_factor, weight_fn):
    out = []
    for e in rows:
        w = weight_fn(e)
        n_copies = max(1, int(base_factor * w))
        out += [e] * n_copies
    return out

def liver_weight(ex):
    k = task_kind(ex)
    if k == 'multi':
        ans = str(ex['output']).strip()
        return STAGE_W.get(ans, 1)
    if k == 'binary':
        return 3 if str(ex['output']).strip().lower() == 'yes' else 1
    if k == 'nas':
        return 3
    return 1

liver_single_up = upsample_weighted(liver_single, 8, liver_weight)
fujiwara_up = list(fujiwara) * 8
dnam_up = list(dnam) * 20 if dnam else []

single = liver_single_up + ext_single + fujiwara_up + dnam_up
pair = liver_pair + ext_pair
drugs = rs.sample(drugs1 + drugs2, 10000)
know = rs.sample(knowledge + dge, 8000)

mixture = single + pair + drugs + know
rs.shuffle(mixture)
json.dump(mixture, open('/content/sft_multitask_17b.json', 'w'), indent=0)
print(f'liver singles: {len(liver_single)} -> {len(liver_single_up)} (class-balanced x8 base)')
print(f'single pool {len(single)} | pair {len(pair)} | drugs {len(drugs)} | know {len(know)}')
print(f'FINAL v4 MIXTURE: {len(mixture)}')
from collections import Counter
mc = [str(e['output']) for e in liver_single_up if task_kind(e) == 'multi']
print('multiclass stage dist after balancing:', dict(sorted(Counter(mc).items())))
bn = [str(e['output']) for e in liver_single_up if task_kind(e) == 'binary']
print('binary yes/no dist:', dict(Counter(bn)))

In [ ]:
# 1.7B config + training (run AFTER the v4 sampler cell above)
import subprocess, sys, os

ds_entry = '''{
  "masld_sft_17b": {
    "file_name": "/content/sft_multitask_17b.json",
    "columns": {"prompt": "instruction", "response": "output"}
  }
}'''
%cd /content/LLaMA-Factory
open('data/dataset_info.json', 'w').write(ds_entry)

CKPT_17B = f'{CKPT}/qwen3-1.7b-masld-sft'
yaml_17b = '''
model_name_or_path: Qwen/Qwen3-1.7B
trust_remote_code: true
stage: sft
do_train: true
finetuning_type: lora
lora_rank: 32
lora_alpha: 32
lora_dropout: 0.0
dataset: masld_sft_17b
template: qwen
cutoff_len: 4096
packing: true
neat_packing: true
max_samples: 100000
overwrite_cache: true
preprocessing_num_workers: 16
output_dir: ''' + CKPT_17B + '''
logging_steps: 10
save_steps: 0.25
plot_loss: true
per_device_train_batch_size: 1
gradient_accumulation_steps: 32
learning_rate: 1.0e-4
num_train_epochs: 2.0
lr_scheduler_type: cosine
warmup_ratio: 0.1
weight_decay: 0.01
bf16: true
gradient_checkpointing: true
enable_liger_kernel: true
'''
open('/content/qwen17b_sft.yaml', 'w').write(yaml_17b)
print('config written; output:', CKPT_17B)

# CUDA fixes (as before)
!pip uninstall -y torchaudio
!pip install torchaudio --index-url https://download.pytorch.org/whl/cu130
!pip -q install liger-kernel
!pip -q install -U 'torchao>=0.16'

# ~10-14h on L4. Checkpoints save to Drive every quarter-epoch (resumable).
!DISABLE_VERSION_CHECK=1 python -m llamafactory.cli train /content/qwen17b_sft.yaml

In [ ]:
# 1.7B merge + eval (run after training completes)
# Direct PeFT merge (skips the flaky llamafactory export), staleness-guarded,
# newline-stopped generation, UNIT-TESTED parser, archive + inline scores.
import os, glob, gzip, json, random, re, subprocess, sys, shutil, datetime
import numpy as np

CKPT_17B = f'{CKPT}/qwen3-1.7b-masld-sft'
MERGED_17B = '/content/qwen3-1.7b-masld-merged'

adapter = None
if os.path.exists(os.path.join(CKPT_17B, 'adapter_model.safetensors')):
    adapter = CKPT_17B
else:
    ckpts = sorted(glob.glob(f'{CKPT_17B}/checkpoint-*'), key=lambda p: int(p.rsplit('-', 1)[-1]))
    if ckpts: adapter = ckpts[-1]
assert adapter, f'no adapter in {CKPT_17B} - did training run?'
if adapter != CKPT_17B:
    print('NOTE: merging latest checkpoint (training interrupted?)')
print('merging adapter:', adapter)

if not os.path.exists(f'{MERGED_17B}/config.json') or \
   os.path.getmtime(f'{MERGED_17B}/model.safetensors' if os.path.exists(f'{MERGED_17B}/model.safetensors') else f'{MERGED_17B}/config.json') < os.path.getmtime(os.path.join(adapter, 'adapter_model.safetensors')):
    print('building fresh merge...')
    import torch
    from transformers import AutoModelForCausalLM, AutoTokenizer
    from peft import PeftModel
    base = AutoModelForCausalLM.from_pretrained('Qwen/Qwen3-1.7B', torch_dtype=torch.bfloat16)
    model = PeftModel.from_pretrained(base, adapter)
    model = model.merge_and_unload()
    model.save_pretrained(MERGED_17B, safe_serialization=True)
    AutoTokenizer.from_pretrained('Qwen/Qwen3-1.7B').save_pretrained(MERGED_17B)
    try:
        dst = f'{CKPT}/qwen3-1.7b-masld-merged'
        if os.path.exists(dst): shutil.rmtree(dst)
        shutil.copytree(MERGED_17B, dst)
        print('persisted to Drive')
    except Exception as e:
        print('Drive copy skipped:', e)
else:
    print('merge is fresh')

# --- eval with the TESTED parser ---
def parse_answer(text, task):
    raw = (text or '').strip()
    t = raw.lower()
    if task.endswith(('001', '201')):
        if t.startswith('yes') or 'answer: yes' in t or 'answer is yes' in t: return 'yes'
        if t.startswith('no') or 'answer: no' in t or 'answer is no' in t: return 'no'
        return None
    if task.endswith(('003', '203')):
        if re.search(r'\banswer(?: is|:) a\b', t) or t.startswith('a'): return 'A'
        if re.search(r'\banswer(?: is|:) b\b', t) or t.startswith('b'): return 'B'
        return None
    if task.endswith(('002', '202')):
        m = re.search(r'F([0-4])', raw)
        if m: return f'F{m.group(1)}'
        m = re.search(r'f([0-4])', t)
        return f'F{m.group(1)}' if m else None
    m = re.search(r'[Ff]([0-4])\b', raw)
    if m: return f'F{m.group(1)}'
    m = re.search(r'([0-9]+(?:\.[0-9]+)?)', t)
    if m:
        try: return float(m.group(1))
        except ValueError: return None
    return None

import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
tok = AutoTokenizer.from_pretrained(MERGED_17B, padding_side='left')
if tok.pad_token is None: tok.pad_token = tok.eos_token
model = AutoModelForCausalLM.from_pretrained(MERGED_17B, torch_dtype=torch.bfloat16, device_map='cuda')
model.eval()

rng = random.Random(20260929)
rows = []
with gzip.open(f'{PROMPTS}/MB-20xx_liver_test.jsonl.gz', 'rt') as f:
    f.readline()
    for line in f: rows.append(json.loads(line))
by_task = {}
for r in rows: by_task.setdefault(r['task'], []).append(r)
sample = []
for task, items in sorted(by_task.items()):
    rng.shuffle(items)
    sample.extend(items[:300])
print('eval prompts:', len(sample))

raws = []
BATCH = 8
i = 0
while i < len(sample):
    batch = sample[i:i+BATCH]
    try:
        enc = tok([r['prompt'] for r in batch], return_tensors='pt',
                  padding=True, truncation=True, max_length=4096).to('cuda')
        with torch.no_grad():
            gen = model.generate(**enc, max_new_tokens=30, do_sample=False,
                                 pad_token_id=tok.pad_token_id)
        for j in range(len(batch)):
            text = tok.decode(gen[j][enc['input_ids'].shape[1]:], skip_special_tokens=True)
            raws.append(text.split('\n')[0])
        i += len(batch)
    except torch.cuda.OutOfMemoryError:
        torch.cuda.empty_cache()
        if BATCH == 1: raise
        BATCH = max(1, BATCH // 2)
        print(f'OOM -> batch {BATCH}')
    if i % 80 < BATCH:
        print(f'  {i}/{len(sample)}', flush=True)

runs_dir = (os.path.join(MASLD_LLM, 'results', 'frontier_runs')
            if os.access(os.path.join(MASLD_LLM, 'results'), os.W_OK)
            else '/content/drive/MyDrive/masld-llm-outputs/frontier_runs')
os.makedirs(runs_dir, exist_ok=True)
run_path = runs_dir + '/qwen3-1.7b-masld__MB-20xx.jsonl.gz'
with gzip.open(run_path, 'wt') as f:
    for r, raw in zip(sample, raws):
        f.write(json.dumps(dict(task=r['task'], sample=r['sample'], answer=r['answer'],
                                raw=raw, error=None, model_version='qwen3-1.7b-masld-v4',
                                parsed=parse_answer(raw, r['task']))) + chr(10))
print('archived ->', run_path)

import collections
bt = collections.defaultdict(list)
for r, raw in zip(sample, raws):
    bt[r['task']].append((parse_answer(raw, r['task']), r['answer']))
print('\n=== qwen3-1.7b-masld v4 ===')
for t, pairs in sorted(bt.items()):
    valid = sum(1 for p, g in pairs if p is not None)
    n = len(pairs)
    if t in ('MB-2001', 'MB-2002', 'MB-2003'):
        acc = sum(1 for p, g in pairs if p == g) / n
        print(f'  {t}: acc={acc:.3f} validity={valid/n:.3f}')
    else:
        nums = [(p, float(g)) for p, g in pairs if isinstance(p, (int, float))]
        if nums:
            mae = float(np.mean([abs(p-g) for p, g in nums]))
            from scipy.stats import spearmanr
            rho = float(spearmanr([g for p,g in nums],[p for p,g in nums]).statistic)
            print(f'  {t}: mae={mae:.3f} rho={rho:.3f} validity={valid/n:.3f}')
        else:
            print(f'  {t}: no parseable, validity={valid/n:.3f}')
print('\nreferences: 0.6B-v3 pairwise 0.593 / multi 0.208 / binary 0.405 | nemotron binary 0.661 / pair 0.500 / multi 0.231')